# GISPR Module 7 — Visualizing Spatial Data: Static Maps to Interactive Web Maps

**Course:** GIS Spatial Analysis with Python and R (GISPR)  
**Module:** 7 — Visualization: Symbology, Classification, Cartography, and Web Maps  
**Builds on:** Module 3 (vector data, CRS), Modules 5–6 (raster data, map algebra, zonal statistics)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Generate the synthetic Nooksack dataset — reaches, subbasins, DEM — with no downloads | Section 1 |
| 2 | Move from a debugging plot to a publication-quality static map | Section 2 |
| 3 | Match the visual variable to the data — and normalize before you map | Section 3 |
| 4 | Apply and defend a classification method; use fixed breaks for comparison maps | Section 4 |
| 5 | Add the cartographic furniture: legend, scale bar, north arrow, credits, CRS | Section 5 |
| 6 | Layer rasters and vectors on one figure without silent CRS misalignment | Section 6 |
| 7 | Publish to the browser — `folium` and `leafmap` (Python), `leaflet` and `mapview` (R) | Section 7 |
| 8 | Drive ArcGIS Pro symbology and layouts from `arcpy.mp`; bridge to R | Section 8 |
| 9 | Build a report-ready map end to end (Lab) | Section 9 |
| 10 | Wrap it in a reusable function you carry into Module 8 | Section 10 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste the command into your terminal / Anaconda Prompt, **not** in a notebook cell

> **The framing problem for this module:**  
> Your Module 6 habitat prioritization runs clean. The reach scores are defensible.  
> Tuesday you're in front of the salmon recovery board. They need **a map in the report** and **a map on the public comment site**.  
> Nobody in that room will read your code. They will judge the map.
>
> The analysis is done. Communicating it is the work that's left — and it is not a lesser kind of work.

> **A warning you should take personally:** every map is an argument. The default settings in every library
> on this page will happily make an argument you did not intend and cannot defend. This module is about
> owning that argument on purpose.

---
## Section 1 — The Data: A Synthetic Factory

Every dataset in this notebook is **generated in code**. Nothing is downloaded, no credentials are needed,
and no path points anywhere outside this notebook's temp folder. That's deliberate: these cells have to run
identically on JupyterHub, on your laptop, and inside an ArcGIS Pro notebook.

We're rebuilding the **Module 6 output**: stream reaches in the Nooksack Basin, each carrying a habitat
`priority` score from 0–100, plus the subbasins they sit in and a DEM for context.

| Layer | Geometry | Key attributes |
|---|---|---|
| `reaches` | LineString | `reach_id`, `subbasin`, `length_km`, `barriers`, `spawners`, `priority` |
| `subbasins` | Polygon | `subbasin`, `reach_count`, `spawners`, `stream_km`, `area_km2` |
| `dem` | Raster (float32) | elevation in metres, 120 m cells |

Two details are in here on purpose, because real data has them:

- **`priority` has four `NaN` values.** Reaches that couldn't be scored. A map that silently drops them
  implies a zero. You'll handle this explicitly in Section 5.
- **The subbasins are deliberately uneven in size.** North Fork is large and sparse; South Fork is small
  and productive. That's what makes Section 3 work.

In [ ]:
# [Python] The synthetic data factory — reaches, subbasins, DEM
# Everything below is generated. No downloads, no credentials, no external paths.

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from rasterio.transform import from_origin

CRS_UTM = "EPSG:32610"      # UTM Zone 10N — metres, correct for NW Washington
OX, OY   = 538000, 5398000  # approximate Nooksack Basin origin


def _wiggly(x0, y0, x1, y1, rng, n=14, amp=420):
    """A straight line with correlated random offsets — looks like a stream reach."""
    t = np.linspace(0, 1, n)
    x, y = x0 + (x1 - x0) * t, y0 + (y1 - y0) * t
    dx, dy = -(y1 - y0), (x1 - x0)
    L = np.hypot(dx, dy) or 1.0
    dx, dy = dx / L, dy / L
    off = np.cumsum(rng.normal(0, amp / 3, n))
    off -= np.linspace(off[0], off[-1], n)       # pin both endpoints back down
    return LineString(np.c_[x + dx * off, y + dy * off])


def make_reaches(seed=7):
    """Stream reaches with a habitat priority score — the Module 6 output."""
    rng = np.random.default_rng(seed)
    rows, rid = [], 1
    trunks = {
        "North Fork":  ((OX + 30000, OY + 26000), (OX + 9000, OY + 6000)),
        "Middle Fork": ((OX + 34000, OY + 14000), (OX + 10000, OY + 5200)),
        "South Fork":  ((OX + 28000, OY + 1500),  (OX + 10500, OY + 4600)),
        "Mainstem":    ((OX + 9000,  OY + 5000),  (OX + 500,  OY + 3000)),
    }
    # Uneven on purpose: North Fork is big and sparse, South Fork small and dense.
    n_tribs = {"North Fork": 9, "Middle Fork": 5, "South Fork": 2, "Mainstem": 4}

    for sb, (a, b) in trunks.items():
        f = np.linspace(0, 1, 7)
        for i in range(len(f) - 1):
            x0, y0 = a[0] + (b[0] - a[0]) * f[i],     a[1] + (b[1] - a[1]) * f[i]
            x1, y1 = a[0] + (b[0] - a[0]) * f[i + 1], a[1] + (b[1] - a[1]) * f[i + 1]
            rows.append((rid, sb, _wiggly(x0, y0, x1, y1, rng), True)); rid += 1
        for _ in range(n_tribs[sb]):
            t = rng.uniform(0.1, 0.9)
            x0, y0 = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
            ang, ln = rng.uniform(0, 2 * np.pi), rng.uniform(3500, 9000)
            rows.append((rid, sb, _wiggly(x0, y0, x0 + ln * np.cos(ang),
                                          y0 + ln * np.sin(ang), rng, amp=260), False)); rid += 1

    gdf = gpd.GeoDataFrame(
        {"reach_id": [r[0] for r in rows],
         "subbasin": [r[1] for r in rows],
         "is_trunk": [r[3] for r in rows]},
        geometry=[r[2] for r in rows], crs=CRS_UTM)

    gdf["length_km"] = gdf.length / 1000
    n = len(gdf)
    gdf["barriers"] = rng.poisson(1.1, n)
    per_sb = {"North Fork": 120, "Middle Fork": 175, "South Fork": 340, "Mainstem": 210}
    base = gdf["subbasin"].map(per_sb).to_numpy()
    gdf["spawners"] = np.clip(rng.normal(base, 55) + gdf["is_trunk"] * 90, 5, None).round()

    raw = (0.5 * (gdf["spawners"]  / gdf["spawners"].max())
         + 0.3 * (gdf["length_km"] / gdf["length_km"].max())
         - 0.2 * (gdf["barriers"]  / max(gdf["barriers"].max(), 1)))
    gdf["priority"] = (100 * (raw - raw.min()) / (raw.max() - raw.min())).round(1)

    # Real data has holes. Four reaches could not be scored.
    gdf.loc[rng.choice(n, 4, replace=False), "priority"] = np.nan
    return gdf.drop(columns="is_trunk")


def make_subbasins(reaches):
    """Subbasin polygons with counts, totals, and area."""
    rows = [{"subbasin": sb, "geometry": grp.union_all().convex_hull.buffer(1800)}
            for sb, grp in reaches.groupby("subbasin")]
    gdf = gpd.GeoDataFrame(rows, crs=CRS_UTM)
    agg = reaches.groupby("subbasin").agg(
        reach_count=("reach_id", "size"),
        spawners=("spawners", "sum"),
        stream_km=("length_km", "sum")).reset_index()
    gdf = gdf.merge(agg, on="subbasin")
    gdf["area_km2"] = gdf.area / 1e6
    return gdf


def make_dem(reaches, res=120, seed=11):
    """Synthetic DEM covering the network. Returns (array, affine transform)."""
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = reaches.total_bounds
    pad = 3000
    minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad
    w, h = int((maxx - minx) / res), int((maxy - miny) / res)
    yy, xx = np.mgrid[0:h, 0:w]
    z = (1500 * np.exp(-(((xx - w * 0.85) ** 2) / (2 * (w * 0.30) ** 2) +
                         ((yy - h * 0.20) ** 2) / (2 * (h * 0.35) ** 2)))
       + 1100 * np.exp(-(((xx - w * 0.70) ** 2) / (2 * (w * 0.22) ** 2) +
                         ((yy - h * 0.85) ** 2) / (2 * (h * 0.28) ** 2)))
       + 55 * np.sin(xx / 7.0) * np.cos(yy / 9.0)
       + np.linspace(260, 20, w)[None, :]
       + rng.normal(0, 6, (h, w)))
    return z.astype("float32"), from_origin(minx, maxy, res, res)


reaches   = make_reaches()
subbasins = make_subbasins(reaches)
dem, dem_transform = make_dem(reaches)

print(f"reaches   : {len(reaches)} features | CRS {reaches.crs.to_string()}")
print(f"subbasins : {len(subbasins)} features")
print(f"dem       : {dem.shape} | {dem.min():.0f}–{dem.max():.0f} m")
print(f"unscored reaches (priority is NaN): {int(reaches['priority'].isna().sum())}")
reaches.head()

In [ ]:
# [R] The synthetic data factory — reaches, subbasins, DEM
# Switch the kernel to R before running. Everything is generated — no downloads.

library(sf)
library(terra)
library(dplyr)

set.seed(7)
CRS_UTM <- 32610          # UTM Zone 10N — metres, correct for NW Washington
OX <- 538000; OY <- 5398000

# A straight line with correlated random offsets — looks like a stream reach
wiggly <- function(x0, y0, x1, y1, n = 14, amp = 420) {
  t  <- seq(0, 1, length.out = n)
  x  <- x0 + (x1 - x0) * t
  y  <- y0 + (y1 - y0) * t
  dx <- -(y1 - y0); dy <- (x1 - x0)
  L  <- sqrt(dx^2 + dy^2); if (L == 0) L <- 1
  dx <- dx / L; dy <- dy / L
  off <- cumsum(rnorm(n, 0, amp / 3))
  off <- off - seq(off[1], off[n], length.out = n)   # pin both endpoints back down
  st_linestring(cbind(x + dx * off, y + dy * off))
}

trunks <- list(
  "North Fork"  = c(OX + 30000, OY + 26000, OX + 9000,  OY + 6000),
  "Middle Fork" = c(OX + 34000, OY + 14000, OX + 10000, OY + 5200),
  "South Fork"  = c(OX + 28000, OY + 1500,  OX + 10500, OY + 4600),
  "Mainstem"    = c(OX + 9000,  OY + 5000,  OX + 500,   OY + 3000)
)
# Uneven on purpose: North Fork is big and sparse, South Fork small and dense.
n_tribs <- c("North Fork" = 9, "Middle Fork" = 5, "South Fork" = 2, "Mainstem" = 4)

geoms <- list(); meta <- list(); rid <- 1
for (sb in names(trunks)) {
  a <- trunks[[sb]]
  f <- seq(0, 1, length.out = 7)
  for (i in 1:(length(f) - 1)) {
    x0 <- a[1] + (a[3] - a[1]) * f[i]
    y0 <- a[2] + (a[4] - a[2]) * f[i]
    x1 <- a[1] + (a[3] - a[1]) * f[i + 1]
    y1 <- a[2] + (a[4] - a[2]) * f[i + 1]
    geoms[[rid]] <- wiggly(x0, y0, x1, y1)
    meta[[rid]]  <- data.frame(reach_id = rid, subbasin = sb, is_trunk = TRUE)
    rid <- rid + 1
  }
  for (k in seq_len(n_tribs[[sb]])) {
    tt  <- runif(1, 0.1, 0.9)
    x0  <- a[1] + (a[3] - a[1]) * tt
    y0  <- a[2] + (a[4] - a[2]) * tt
    ang <- runif(1, 0, 2 * pi); ln <- runif(1, 3500, 9000)
    geoms[[rid]] <- wiggly(x0, y0, x0 + ln * cos(ang), y0 + ln * sin(ang), amp = 260)
    meta[[rid]]  <- data.frame(reach_id = rid, subbasin = sb, is_trunk = FALSE)
    rid <- rid + 1
  }
}

reaches <- st_sf(do.call(rbind, meta), geometry = st_sfc(geoms, crs = CRS_UTM))
reaches$length_km <- as.numeric(st_length(reaches)) / 1000
reaches$barriers  <- rpois(nrow(reaches), 1.1)

# South Fork reaches are short but productive; North Fork is long but sparse.
per_sb <- c("North Fork" = 120, "Middle Fork" = 175, "South Fork" = 340, "Mainstem" = 210)
base   <- as.numeric(per_sb[reaches$subbasin])
reaches$spawners <- pmax(round(rnorm(nrow(reaches), base, 55) + reaches$is_trunk * 90), 5)

raw <- 0.5 * (reaches$spawners  / max(reaches$spawners)) +
       0.3 * (reaches$length_km / max(reaches$length_km)) -
       0.2 * (reaches$barriers  / max(reaches$barriers))
reaches$priority <- round(100 * (raw - min(raw)) / (max(raw) - min(raw)), 1)
reaches$priority[sample(nrow(reaches), 4)] <- NA   # real data has holes

subbasins <- reaches |>
  group_by(subbasin) |>
  summarise(reach_count = n(),
            spawners    = sum(spawners),
            stream_km   = sum(length_km)) |>
  st_convex_hull() |>
  st_buffer(1800)
subbasins$area_km2 <- as.numeric(st_area(subbasins)) / 1e6

# ── DEM via terra ───────────────────────────────────────────────────────────
bb <- st_bbox(reaches)
r  <- rast(xmin = bb[["xmin"]] - 3000, xmax = bb[["xmax"]] + 3000,
           ymin = bb[["ymin"]] - 3000, ymax = bb[["ymax"]] + 3000,
           resolution = 120, crs = paste0("EPSG:", CRS_UTM))
xy <- xyFromCell(r, 1:ncell(r))
xn <- (xy[, 1] - xmin(r)) / (xmax(r) - xmin(r))
yn <- (xy[, 2] - ymin(r)) / (ymax(r) - ymin(r))
values(r) <- 1500 * exp(-(((xn - 0.85)^2) / (2 * 0.30^2) + ((yn - 0.80)^2) / (2 * 0.35^2))) +
             1100 * exp(-(((xn - 0.70)^2) / (2 * 0.22^2) + ((yn - 0.15)^2) / (2 * 0.28^2))) +
             260 * (1 - xn) + rnorm(ncell(r), 0, 6)
names(r) <- "elevation"

cat("reaches  :", nrow(reaches), "features | CRS:", st_crs(reaches)$epsg, "\n")
cat("subbasins:", nrow(subbasins), "features\n")
cat("dem      :", nrow(r), "x", ncol(r), "|",
    round(minmax(r)[1]), "-", round(minmax(r)[2]), "m\n")
cat("unscored reaches (priority is NA):", sum(is.na(reaches$priority)), "\n")
head(reaches)

### 🔧 Try it yourself — the data

1. Change the `seed` in `make_reaches()` and re-run. How much does the `priority` distribution move?
   (`reaches["priority"].describe()`)
2. Print the count of reaches per subbasin (`reaches["subbasin"].value_counts()`). Which subbasin has the
   most reaches? Hold on to that answer — Section 3 is going to complicate it.
3. Add a `barriers_per_km` column. Is it correlated with `priority`? Should it be, given how the score
   was built?

---
## Section 2 — From Debugging Plot to Map

`gdf.plot()` is not a map. It's a debugging view — and it is the single most common thing that accidentally
ends up pasted into a board packet.

ArcGIS Pro is *opinionated*: drop a layer in and you get a frame, a coordinate system, and a Contents pane
nagging you toward a legend. **matplotlib gives you an Axes and nothing else.** That's the trade of working
in code — total control, zero guardrails.

So build the map in layers, and watch what each one buys you:

| Step | Call | What it buys |
|---|---|---|
| 1 | `.plot()` | Do the geometries exist? Are they where you expect? |
| 2 | `+ column=` | The attribute becomes visible |
| 3 | `+ scheme=`, `k=` | Continuous values become classes |
| 4 | `+ legend`, title, credits | A reader can now verify your claim |

Steps 1–2 are for you. Steps 3–4 are for everyone else. **Ship step 4.**

In [ ]:
# [Python] Build the map in four layers — run and watch each panel improve

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 12))

# ── 1. The debugging view — this is what .plot() gives you ───────────────────
reaches.plot(ax=axes[0, 0], linewidth=1.2)
axes[0, 0].set_title("1. reaches.plot()\ngrey lines on a white void", loc="left")

# ── 2. Encode the attribute ─────────────────────────────────────────────────
reaches.plot(ax=axes[0, 1], column="priority", cmap="viridis", linewidth=1.6)
axes[0, 1].set_title("2. + column='priority'\nnow it means something — but what?", loc="left")

# ── 3. Classify it ──────────────────────────────────────────────────────────
reaches.plot(ax=axes[1, 0], column="priority", cmap="viridis",
             scheme="natural_breaks", k=5, linewidth=1.6)
axes[1, 0].set_title("3. + scheme='natural_breaks', k=5\ncontinuous -> five classes", loc="left")

# ── 4. Make it readable ─────────────────────────────────────────────────────
ax = axes[1, 1]
reaches.plot(ax=ax, column="priority", cmap="viridis", scheme="natural_breaks", k=5,
             linewidth=1.8, legend=True,
             legend_kwds={"title": "Habitat priority (0–100)", "loc": "lower right",
                          "fontsize": 8, "title_fontsize": 9},
             missing_kwds={"color": "0.75", "label": "Not scored"})
ax.set_title("4. + legend, title, credits\na map a stranger can read", loc="left")
ax.annotate("Data: WDFW 2024 (synthetic) · EPSG:32610 · Analysis: GISPR Module 6",
            xy=(0.01, 0.01), xycoords="axes fraction", fontsize=7, color="0.35")

for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
    for side in a.spines.values():
        side.set_visible(False)

plt.tight_layout()
plt.show()

print("Panel 1 is a debugging view. Panel 4 is a map. The difference is about six lines of code.")

In [ ]:
# [R] The same four layers with sf + tmap

library(sf)
library(tmap)

tmap_mode("plot")   # static

# ⚠ VERSION NOTE: these cells use tmap v3 aesthetic syntax (palette=, style=, n=).
#   tmap v4 accepts it but prints "v3 code detected". Check packageVersion("tmap").

# 1. The debugging view
p1 <- tm_shape(reaches) + tm_lines() +
      tm_layout(title = "1. plot(st_geometry(reaches))", frame = FALSE)

# 2. Encode the attribute
p2 <- tm_shape(reaches) +
      tm_lines(col = "priority", palette = "viridis") +
      tm_layout(title = "2. + col = 'priority'", frame = FALSE)

# 3. Classify it
p3 <- tm_shape(reaches) +
      tm_lines(col = "priority", palette = "viridis", style = "jenks", n = 5) +
      tm_layout(title = "3. + style = 'jenks', n = 5", frame = FALSE)

# 4. Make it readable
p4 <- tm_shape(reaches) +
      tm_lines(col = "priority", palette = "viridis", style = "jenks", n = 5,
               lwd = 2, title.col = "Habitat priority (0-100)",
               colorNA = "grey75", textNA = "Not scored") +
      tm_credits("Data: WDFW 2024 (synthetic) | EPSG:32610",
                 size = 0.5, position = c("left", "bottom")) +
      tm_layout(title = "4. + legend, credits", frame = FALSE,
                legend.position = c("right", "bottom"))

tmap_arrange(p1, p2, p3, p4, nrow = 2)

# ggplot2 equivalent of panel 4 — when you need map + non-map panels in one figure
# library(ggplot2)
# ggplot(reaches) +
#   geom_sf(aes(colour = priority), linewidth = 0.8) +
#   scale_colour_viridis_c(name = "Habitat priority", na.value = "grey75") +
#   labs(caption = "Data: WDFW 2024 (synthetic) | EPSG:32610") +
#   theme_void()

### 🔧 Try it yourself — building in layers

1. Delete `legend=True` from panel 4 and look at it again. How would a reader know whether yellow is
   good or bad?
2. Swap `cmap="viridis"` for `cmap="jet"`. Look at panel 4 hard. Where do you *see* class boundaries
   that the classification never created? (This is the rainbow-ramp problem, and it's why `jet` is
   effectively banned in scientific cartography.)
3. Remove `missing_kwds`. The four unscored reaches vanish. What does a reader now assume about them?

---
## Section 3 — Symbology: Match the Visual Variable to the Data

Symbology is the decision about **what you encode and how**. Get the pairing wrong and the map lies —
fluently, and with a straight face.

| Your data is… | Use | Not |
|---|---|---|
| A rate, ratio, or density (per km², per capita) | Choropleth — fill colour | — |
| A raw count or magnitude (totals, sums) | **Graduated symbols** sized by value | ❌ Choropleth — you'll map polygon *size* |
| Categories with no order (land cover, owner) | Unique values — distinct hues | ❌ A sequential ramp — invents a ranking |
| An ordered sequence (low → high) | Sequential ramp (`viridis`, `YlGnBu`) | ❌ A rainbow ramp — invents breaks |
| Divergence around a meaningful midpoint | Diverging ramp (`RdBu`), midpoint pinned | ❌ Sequential — hides the midpoint |

### The one that gets everyone

**Never map a raw count as a choropleth.** A big polygon has more of *everything* — more people, more
permits, more stream reaches — because it's *big*. Colour the polygon by the count and you have drawn a
map of polygon area with extra steps.

Our subbasins make this concrete. Run the next cell.

In [ ]:
# [Python] Raw count vs. normalized — the same data, the opposite conclusion

summary = subbasins[["subbasin", "reach_count", "area_km2", "spawners"]].copy()
summary["spawners_per_km2"] = (summary["spawners"] / summary["area_km2"]).round(1)

print(summary.sort_values("reach_count", ascending=False).to_string(index=False))
print()
print("Ranked by RAW COUNT      :", list(summary.sort_values("reach_count", ascending=False)["subbasin"]))
print("Ranked by SPAWNER DENSITY:", list(summary.sort_values("spawners_per_km2", ascending=False)["subbasin"]))
print()
print("North Fork is FIRST by count and LAST by density.")
print("Two defensible maps. Two opposite recommendations to the board. Same data.")

In [ ]:
# [Python] Draw both maps side by side — and one honest alternative

sb = subbasins.copy()
sb["spawners_per_km2"] = sb["spawners"] / sb["area_km2"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))

# ── WRONG: raw count as a choropleth ────────────────────────────────────────
sb.plot(ax=axes[0], column="spawners", cmap="YlGnBu", edgecolor="white",
        linewidth=0.8, legend=True, legend_kwds={"shrink": 0.6})
axes[0].set_title("✗ WRONG — raw spawner count\nas a choropleth", loc="left", color="#9E2F27")

# ── RIGHT: normalize first ──────────────────────────────────────────────────
sb.plot(ax=axes[1], column="spawners_per_km2", cmap="YlGnBu", edgecolor="white",
        linewidth=0.8, legend=True, legend_kwds={"shrink": 0.6})
axes[1].set_title("✓ RIGHT — spawners per km²\n(normalized)", loc="left", color="#1B5C46")

# ── ALSO RIGHT: keep the count, change the visual variable ──────────────────
sb.plot(ax=axes[2], facecolor="0.94", edgecolor="white", linewidth=0.8)
cent = sb.copy(); cent["geometry"] = cent.geometry.representative_point()
cent.plot(ax=axes[2], markersize=cent["spawners"] / 6, color="#4B2E83", alpha=0.75)
axes[2].set_title("✓ ALSO RIGHT — graduated symbols\nsized by raw count", loc="left", color="#1B5C46")

for a, lab in zip(axes, ["spawners", "spawners_per_km2", None]):
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
    for _, row in sb.iterrows():
        p = row.geometry.representative_point()
        a.annotate(row["subbasin"], (p.x, p.y), ha="center", fontsize=7.5, color="0.25")

plt.tight_layout(); plt.show()

print("Left and centre use the SAME numbers and disagree about which subbasin matters most.")
print("Only one of them answers the question the board is actually asking.")

In [ ]:
# [R] Raw count vs. normalized with tmap

library(tmap)
library(dplyr)

subbasins$spawners_per_km2 <- round(subbasins$spawners / subbasins$area_km2, 1)

# ── The ranking check — the same inversion you saw in Python ────────────────
tab <- st_drop_geometry(subbasins)[, c("subbasin", "reach_count", "area_km2",
                                       "spawners", "spawners_per_km2")]
print(tab[order(-tab$reach_count), ], row.names = FALSE)
cat("\nRanked by RAW COUNT      :",
    paste(tab$subbasin[order(-tab$reach_count)], collapse = ", "), "\n")
cat("Ranked by SPAWNER DENSITY:",
    paste(tab$subbasin[order(-tab$spawners_per_km2)], collapse = ", "), "\n")
cat("\nNorth Fork is FIRST by count and LAST by density.\n")
cat("Two defensible maps. Two opposite recommendations. Same data.\n")

# ── ✗ WRONG — raw count as a choropleth ────────────────────────────────────
m_wrong <- tm_shape(subbasins) +
  tm_polygons(col = "spawners", palette = "YlGnBu", style = "quantile",
              title = "Spawners (count)") +
  tm_layout(title = "WRONG - raw count choropleth", frame = FALSE)

# ── ✓ RIGHT — normalize before you map ─────────────────────────────────────
m_right <- tm_shape(subbasins) +
  tm_polygons(col = "spawners_per_km2", palette = "YlGnBu", style = "quantile",
              title = "Spawners per km2") +
  tm_layout(title = "RIGHT - normalized", frame = FALSE)

# ── ✓ ALSO RIGHT — keep the count, change the visual variable ───────────────
cent <- st_point_on_surface(subbasins)
m_symb <- tm_shape(subbasins) +
  tm_polygons(col = "grey94", border.col = "white") +
  tm_shape(cent) +
  tm_symbols(size = "spawners", col = "#4B2E83", alpha = 0.75,
             title.size = "Spawners (count)") +
  tm_layout(title = "ALSO RIGHT - graduated symbols", frame = FALSE)

tmap_arrange(m_wrong, m_right, m_symb, nrow = 1)

### 🔧 Try it yourself — symbology

1. Map `reach_count` as a choropleth, then map `reach_count / area_km2`. Does the ranking flip the same
   way `spawners` did?
2. The graduated-symbol panel divides by 6 (`markersize=cent["spawners"] / 6`). That divisor is a
   cartographic decision, not a technicality — try 2 and 20. At what point does the map stop being readable?
3. **The professional question:** you're presenting to the board and you have both maps. Which do you show,
   and what exactly do you say when someone asks why the other one looks different? Write the two sentences
   out. That's the deliverable in Section 9.

---
## Section 4 — Classification: Same Data, Five Stories

Classification is how continuous values become classes. **The method is an argument you are making on the
reader's behalf** — usually without telling them.

| Method | `scheme=` / `style=` | What it does | Use it when | The catch |
|---|---|---|---|---|
| Equal interval | `equal_interval` / `equal` | Cuts the *range* into n equal bins | Values are evenly spread; bins are intuitive | Skewed data → most features in one class |
| Quantile | `quantile` / `quantile` | Equal *count* per class | You want visual contrast | Always looks good. Invents differences where none exist |
| Natural breaks | `natural_breaks` / `jenks` | Minimizes within-class variance | Clustered data, single map | **Breaks are computed from this dataset only** |
| Standard deviation | `std_mean` / `sd` | Classes by distance from the mean | Showing deviation from typical | Needs a roughly normal distribution |
| Manual / fixed | `user_defined` / `fixed` | You supply the breaks | Regulatory thresholds; **any comparison** | You have to justify them — which is the point |

### The one that gets everyone

**Jenks is not "the best."** Jenks minimizes within-class variance *for the dataset you handed it*. That is
not the same as being the most honest, and it is actively wrong for comparison.

> **Comparing two maps — two years, two basins, before/after?**
> The breaks must be **identical and manual**. Natural breaks recomputed per map means the two legends
> mean different things, and the comparison is meaningless. It will look completely fine. Nobody will catch
> it in review. That's what makes it dangerous.

In [ ]:
# [Python] The same field, five methods, five stories

import mapclassify
import numpy as np

vals = reaches["priority"].dropna()

methods = {
    "Equal interval":  mapclassify.EqualInterval(vals, k=5),
    "Quantile":        mapclassify.Quantiles(vals, k=5),
    "Natural breaks":  mapclassify.NaturalBreaks(vals, k=5),
    "Std. deviation":  mapclassify.StdMean(vals),
    "Manual (policy)": mapclassify.UserDefined(vals, bins=[20, 40, 60, 80, 100]),
}

for name, m in methods.items():
    breaks = ", ".join(f"{b:.1f}" for b in m.bins)
    print(f"{name:16} breaks: {breaks}")
    print(f"{'':16} counts: {list(m.counts)}")
print()
print("Same 40 reaches. Five different sets of breaks. Five different maps.")
print("Any of them can be defended. Only one of them is the one you meant.")

In [ ]:
# [Python] Draw all five — this is the cell to look at, not the numbers above

fig, axes = plt.subplots(1, 5, figsize=(19, 4.6))

specs = [("Equal interval", "equal_interval", {}),
         ("Quantile", "quantiles", {}),
         ("Natural breaks", "natural_breaks", {}),
         ("Std. deviation", "std_mean", {}),
         ("Manual (policy)", "user_defined", {"classification_kwds": {"bins": [20, 40, 60, 80, 100]}})]

for ax, (title, scheme, extra) in zip(axes, specs):
    reaches.plot(ax=ax, column="priority", cmap="viridis", scheme=scheme, k=5,
                 linewidth=1.7, missing_kwds={"color": "0.8"}, **extra)
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

plt.suptitle("One field. Five classification methods. Look at the North Fork in each.",
             y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

print("If you can see the North Fork change character across these panels — that's the lesson.")
print("Nothing about the data changed. Only the argument did.")

In [ ]:
# [Python] Fixed breaks: the only defensible choice for a comparison map
# Scenario: the board wants North Fork vs South Fork, side by side.

nf = reaches[reaches["subbasin"] == "North Fork"]
sf = reaches[reaches["subbasin"] == "South Fork"]

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

# ── ✗ WRONG — natural breaks recomputed per panel ───────────────────────────
for ax, (gdf, name) in zip(axes[0], [(nf, "North Fork"), (sf, "South Fork")]):
    gdf.plot(ax=ax, column="priority", cmap="viridis", scheme="natural_breaks", k=4,
             linewidth=2.2, legend=True, legend_kwds={"fontsize": 7, "loc": "lower left"})
    ax.set_title(f"✗ {name} — jenks, recomputed", loc="left", color="#9E2F27", fontsize=10)

# ── ✓ RIGHT — identical manual breaks, shared legend meaning ────────────────
BREAKS = [25, 50, 75, 100]
for ax, (gdf, name) in zip(axes[1], [(nf, "North Fork"), (sf, "South Fork")]):
    gdf.plot(ax=ax, column="priority", cmap="viridis", scheme="user_defined",
             classification_kwds={"bins": BREAKS}, linewidth=2.2,
             legend=True, legend_kwds={"fontsize": 7, "loc": "lower left"})
    ax.set_title(f"✓ {name} — fixed breaks {BREAKS}", loc="left", color="#1B5C46", fontsize=10)

for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

# The breaks that were actually used in the top row:
print("North Fork jenks breaks:", np.round(mapclassify.NaturalBreaks(nf["priority"].dropna(), k=4).bins, 1))
print("South Fork jenks breaks:", np.round(mapclassify.NaturalBreaks(sf["priority"].dropna(), k=4).bins, 1))
print()
print("Top row: the same colour means a different number in each panel. The comparison is meaningless.")
print("Bottom row: the same colour means the same number. Now you can compare.")

In [ ]:
# [R] Classification with tmap — same methods, same names

library(tmap)
library(classInt)

tmap_mode("plot")

# The breaks any method would pick — inspect before you commit
vals <- reaches$priority[!is.na(reaches$priority)]
for (s in c("equal", "quantile", "jenks", "sd")) {
  brks <- classIntervals(vals, n = 5, style = s)$brks
  cat(sprintf("%-9s breaks: %s\n", s, paste(round(brks, 1), collapse = ", ")))
}
cat("\nSame reaches. Four different sets of breaks. Four different maps.\n")

styles <- c("equal", "quantile", "jenks", "sd")
maps <- lapply(styles, function(s) {
  tm_shape(reaches) +
    tm_lines(col = "priority", palette = "viridis", style = s, n = 5, lwd = 2,
             colorNA = "grey80") +
    tm_layout(title = paste0("style = '", s, "'"), frame = FALSE, legend.show = FALSE)
})

# Manual / fixed breaks — style = "fixed" + explicit breaks
m_fixed <- tm_shape(reaches) +
  tm_lines(col = "priority", palette = "viridis", style = "fixed",
           breaks = c(0, 20, 40, 60, 80, 100), lwd = 2, colorNA = "grey80") +
  tm_layout(title = "style = 'fixed' (policy)", frame = FALSE, legend.show = FALSE)

do.call(tmap_arrange, c(maps, list(m_fixed), list(nrow = 1)))

# ── The comparison map: identical breaks in BOTH panels ────────────────────
BREAKS <- c(0, 25, 50, 75, 100)
nf  <- reaches[reaches$subbasin == "North Fork", ]
sfk <- reaches[reaches$subbasin == "South Fork", ]

m_nf <- tm_shape(nf) +
  tm_lines(col = "priority", palette = "viridis", style = "fixed",
           breaks = BREAKS, lwd = 2.5) +
  tm_layout(title = "North Fork - fixed breaks", frame = FALSE)
m_sf <- tm_shape(sfk) +
  tm_lines(col = "priority", palette = "viridis", style = "fixed",
           breaks = BREAKS, lwd = 2.5) +
  tm_layout(title = "South Fork - fixed breaks", frame = FALSE)
tmap_arrange(m_nf, m_sf, nrow = 1)

# Jenks recomputed per subbasin — how far apart are the breaks?
cat("\nNorth Fork jenks:",
    paste(round(classIntervals(nf$priority[!is.na(nf$priority)], 4, style = "jenks")$brks, 1),
          collapse = ", "), "\n")
cat("South Fork jenks:",
    paste(round(classIntervals(sfk$priority[!is.na(sfk$priority)], 4, style = "jenks")$brks, 1),
          collapse = ", "), "\n")
cat("That gap is the size of the error in a jenks comparison map.\n")

### 🔧 Try it yourself — classification

1. Change `k=5` to `k=3` and `k=7` in the five-panel figure. Which methods are most sensitive to the class
   count? (Watch quantile.)
2. Run `mapclassify.Quantiles(vals, k=5).counts` — every class has the same count *by construction*. Now
   explain why a quantile map always looks well-balanced even when the data isn't.
3. Compute Jenks breaks for the North Fork alone and for the whole basin. How far apart are they? That gap
   is the size of the error in the ✗ panels above.
4. **The defensible answer:** pick the method you'd take to the board and write one sentence starting
   "I used ___ breaks because ___." If your sentence is "because it looked best," try again.

---
## Section 5 — Cartographic Elements: What Makes a Map Verifiable

A map is a **claim about the world**. These elements are what let a reader check the claim instead of
taking it on trust.

| Element | Why it's non-negotiable |
|---|---|
| **Legend with units** | "Priority" means nothing. "Habitat priority (0–100, WDFW 2024 method)" is checkable. |
| **The NoData class** | If 4 of 44 reaches have no score, that must be *visible*. Silence implies zero. |
| **Title** | States the claim in words, so the reader knows what they're looking at. |
| **Scale bar** | Distance. Required for print; largely redundant on a zoomable web map. |
| **North arrow** | Only when rotation isn't obvious. A north-up map doesn't need one. |
| **Credits: source, vintage, CRS, author** | This is what makes the map survive the room. When someone asks "what projection is this, and does it distort area?" — the answer is already on the map. |

ArcGIS Pro's Layout view hands you most of this for free. In matplotlib **every one of these is a line of
code you have to write.** That's the cost of the control.

In [ ]:
# [Python] The full furniture — a reusable helper you carry into Module 8

from matplotlib.patches import Rectangle

def add_scale_bar(ax, length_m=10000, label=None, loc=(0.06, 0.06), height_frac=0.008):
    """Draw a simple scale bar. Assumes a projected CRS in metres."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    w, h = x1 - x0, y1 - y0
    bx, by = x0 + loc[0] * w, y0 + loc[1] * h
    bh = h * height_frac
    # two-tone bar, the cartographic convention
    ax.add_patch(Rectangle((bx, by), length_m / 2, bh, facecolor="black", edgecolor="black", zorder=5))
    ax.add_patch(Rectangle((bx + length_m / 2, by), length_m / 2, bh,
                           facecolor="white", edgecolor="black", zorder=5))
    ax.text(bx + length_m / 2, by + bh * 2.1, label or f"{length_m/1000:.0f} km",
            ha="center", fontsize=8, zorder=5)

def add_north_arrow(ax, loc=(0.94, 0.90)):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x, y = x0 + loc[0] * (x1 - x0), y0 + loc[1] * (y1 - y0)
    ax.annotate("N", xy=(x, y), xytext=(x, y - 0.07 * (y1 - y0)),
                arrowprops=dict(facecolor="black", width=3.5, headwidth=10),
                ha="center", va="center", fontsize=11, fontweight="bold", zorder=5)

def add_credits(ax, text):
    ax.annotate(text, xy=(0.005, -0.02), xycoords="axes fraction",
                fontsize=7, color="0.35", va="top")


fig, ax = plt.subplots(figsize=(10, 10))

subbasins.plot(ax=ax, facecolor="0.96", edgecolor="0.8", linewidth=0.7)
reaches.plot(ax=ax, column="priority", cmap="viridis", scheme="user_defined",
             classification_kwds={"bins": [20, 40, 60, 80, 100]},
             linewidth=2.2, legend=True,
             legend_kwds={"title": "Habitat priority\n(0–100, WDFW 2024 method)",
                          "loc": "lower right", "fontsize": 8, "title_fontsize": 9,
                          "frameon": True, "framealpha": 0.9},
             missing_kwds={"color": "#C9524A", "label": "Not scored (4 reaches)"})

ax.set_title("Salmon Habitat Priority — Nooksack Basin Reaches",
             fontsize=15, fontweight="bold", loc="left", pad=12)
ax.text(0, 1.005, "Reaches ranked for restoration investment · 2024 assessment",
        transform=ax.transAxes, fontsize=9.5, color="0.4")

add_scale_bar(ax, 10000)
add_north_arrow(ax)
add_credits(ax, "Data: WDFW 2024 (synthetic teaching data) · CRS: EPSG:32610 (UTM 10N, metres) · "
                "Classification: manual breaks at 20/40/60/80 · Analysis: GISPR Module 6 · M. Weber, US EPA")

ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)

plt.tight_layout()
plt.show()

print("Every element above answers a question a reviewer would otherwise have to ask you in person.")

In [ ]:
# [Python] Export at print resolution — then actually open the file

import tempfile, os
OUTDIR = tempfile.mkdtemp(prefix="gispr_m7_")

fig.savefig(os.path.join(OUTDIR, "priority_map_300dpi.png"), dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(OUTDIR, "priority_map_screen.png"), dpi=96,
            bbox_inches="tight", facecolor="white")
# Vector output — infinitely scalable, the right answer for a printed report
fig.savefig(os.path.join(OUTDIR, "priority_map.pdf"), bbox_inches="tight", facecolor="white")

for f in sorted(os.listdir(OUTDIR)):
    print(f"{f:32} {os.path.getsize(os.path.join(OUTDIR, f)):>9,} bytes")

print(f"\nOutput folder: {OUTDIR}")
print("\nOpen the 300 dpi PNG at 100%. Is the legend text legible? Is the scale bar label?")
print("Screen-legible is not print-legible — 8pt legend text disappears on paper.")

In [ ]:
# [R] Cartographic elements with tmap — the furniture is built in

library(tmap)

tmap_mode("plot")

# ⚠ VERSION TRAP: tmap v3 called this tm_scale_bar(); v4 renamed it tm_scalebar().
#   This shim keeps the notebook running on either. Check packageVersion("tmap").
tm_scalebar_c <- if (utils::packageVersion("tmap") >= "4.0.0") {
  tmap::tm_scalebar
} else {
  tmap::tm_scale_bar
}

m <- tm_shape(subbasins) +
       tm_polygons(col = "grey96", border.col = "grey80") +
     tm_shape(reaches) +
       tm_lines(col = "priority", palette = "viridis",
                style = "fixed", breaks = c(0, 20, 40, 60, 80, 100), lwd = 2.2,
                title.col = "Habitat priority\n(0-100, WDFW 2024 method)",
                colorNA = "#C9524A", textNA = "Not scored (4 reaches)") +
     tm_scalebar_c(breaks = c(0, 5, 10), position = c("left", "bottom")) +
     tm_compass(type = "arrow", position = c("right", "top")) +
     tm_credits("Data: WDFW 2024 (synthetic) | CRS: EPSG:32610 (UTM 10N) | GISPR Module 6",
                size = 0.5, position = c("left", "bottom")) +
     tm_layout(title = "Salmon Habitat Priority - Nooksack Basin Reaches",
               frame = FALSE, legend.position = c("right", "bottom"),
               legend.bg.color = "white", legend.bg.alpha = 0.9)
m

# ── Export at print resolution ─────────────────────────────────────────────
outdir <- tempdir()
tmap_save(m, file.path(outdir, "priority_map_300dpi.png"), dpi = 300, width = 8, height = 8)
tmap_save(m, file.path(outdir, "priority_map.pdf"), width = 8, height = 8)   # vector
cat("Saved to:", outdir, "\n")
cat("Open the 300 dpi PNG at 100%. Is the legend legible? The scale bar label?\n")

# ggplot2 equivalent, when you need map + chart panels in one figure
# library(ggplot2); library(ggspatial)
# ggplot() +
#   geom_sf(data = subbasins, fill = "grey96", colour = "grey80") +
#   geom_sf(data = reaches, aes(colour = priority), linewidth = 0.9) +
#   scale_colour_viridis_c(name = "Habitat priority", na.value = "#C9524A") +
#   annotation_scale(location = "bl") +
#   annotation_north_arrow(location = "tr", style = north_arrow_minimal()) +
#   labs(title = "Salmon Habitat Priority - Nooksack Basin",
#        caption = "Data: WDFW 2024 (synthetic) | EPSG:32610") +
#   theme_void()

### 🔧 Try it yourself — cartographic elements

1. Open `priority_map_300dpi.png` at 100% zoom. Now open the 96 dpi version. Which one could you hand to a
   print shop?
2. Change `missing_kwds` colour from red to `"0.75"` (grey). Which choice is more honest for a board
   packet — loud red, or quiet grey? Defend it. (There isn't one right answer; there is a right *reason*.)
3. Delete `add_credits(...)` and imagine the board asks "is this in feet or metres, and how old is this
   data?" Every element you remove is a question you have to answer live, from memory.
4. Add a `barriers` overlay as point symbols at reach midpoints. Does the map get more informative, or
   just busier? Cartography is subtraction as often as addition.

---
## Section 6 — Layering Rasters and Vectors

Your Module 5–6 rasters and your Module 3–4 vectors, on one figure. Two rules:

1. **Draw order is call order.** Hillshade first, at `alpha` 0.5–0.6. Vectors on top. Reverse them and your
   reaches vanish — which looks like a data problem but is a z-order problem.
2. **Align the CRS *before* the first plot call.**

> ⚠ **matplotlib is not a GIS.** It plots *coordinates*, not geography. Hand it two layers in different
> CRS and it will render them into the same Axes without a word of complaint. The result looks like a
> strange projection artifact rather than a bug. ArcGIS Pro reprojects on the fly and hides this from you —
> in code, alignment is your job. This is a real cost of leaving the ESRI stack, and it's honest to say so.

In [ ]:
# [Python] Hillshade + reaches — and the CRS trap, demonstrated

import rasterio
from rasterio.plot import show as rshow

def hillshade(dem, az=315, alt=45, res=120):
    """Standard hillshade from a DEM array. Carry this forward — Module 8 uses it."""
    x, y = np.gradient(dem, res, res)
    slope  = np.pi / 2 - np.arctan(np.hypot(x, y))
    aspect = np.arctan2(-x, y)
    azr, altr = np.radians(360 - az + 90), np.radians(alt)
    hs = (np.sin(altr) * np.sin(slope) +
          np.cos(altr) * np.cos(slope) * np.cos(azr - aspect))
    return (255 * (hs + 1) / 2).astype("uint8")

hs = hillshade(dem)

# Write the DEM to a real GeoTIFF so the rest of the notebook can read it like any raster
dem_path = os.path.join(OUTDIR, "nooksack_dem.tif")
with rasterio.open(dem_path, "w", driver="GTiff", height=dem.shape[0], width=dem.shape[1],
                   count=1, dtype="float32", crs=CRS_UTM, transform=dem_transform,
                   nodata=-9999, compress="deflate") as dst:
    dst.write(dem, 1)

hs_path = os.path.join(OUTDIR, "nooksack_hillshade.tif")
with rasterio.open(hs_path, "w", driver="GTiff", height=hs.shape[0], width=hs.shape[1],
                   count=1, dtype="uint8", crs=CRS_UTM, transform=dem_transform,
                   compress="deflate") as dst:
    dst.write(hs, 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

# ── ✓ RIGHT — same CRS, raster first, vectors on top ────────────────────────
ax = axes[0]
with rasterio.open(hs_path) as src:
    rshow(src, ax=ax, cmap="Greys_r", alpha=0.55)
subbasins.plot(ax=ax, facecolor="none", edgecolor="0.45", linewidth=0.7, linestyle="--")
reaches.plot(ax=ax, column="priority", cmap="viridis", scheme="user_defined",
             classification_kwds={"bins": [20, 40, 60, 80, 100]},
             linewidth=2.0, missing_kwds={"color": "#C9524A"})
ax.set_title("✓ RIGHT — hillshade (alpha 0.55), then vectors on top",
             loc="left", color="#1B5C46")

# ── ✗ WRONG — reaches reprojected to 4326, raster left in UTM ───────────────
ax = axes[1]
with rasterio.open(hs_path) as src:
    rshow(src, ax=ax, cmap="Greys_r", alpha=0.55)
reaches.to_crs(4326).plot(ax=ax, color="#C9524A", linewidth=2.0)   # degrees vs metres!
ax.set_title("✗ WRONG — reaches in EPSG:4326, raster in EPSG:32610\n"
             "matplotlib did not warn you", loc="left", color="#9E2F27")

for a in axes:
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

print("Right panel: the reaches are drawn near (-122, 48). The raster spans (538000, 5398000).")
print("They're both on the Axes. Neither library raised a single warning.")
print("\nThe defensive habit — assert, don't hope:")
print(f"  reaches.crs == raster.crs ->  {reaches.crs == rasterio.open(hs_path).crs}")

In [ ]:
# [Python] The defensive check — steal this for every multi-layer script

def assert_aligned(*layers, target=None):
    """Raise loudly if any layer's CRS disagrees. Call before the first plot."""
    crs_list = []
    for lyr in layers:
        crs_list.append(lyr.crs if hasattr(lyr, "crs") else rasterio.open(lyr).crs)
    target = target or crs_list[0]
    bad = [i for i, c in enumerate(crs_list) if c != target]
    if bad:
        raise ValueError(
            f"CRS mismatch in layers {bad}: {[str(crs_list[i]) for i in bad]} != {target}")
    print(f"✓ all {len(layers)} layers aligned on {target.to_string() if hasattr(target,'to_string') else target}")
    return target

# Passes
assert_aligned(reaches, subbasins, hs_path)

# Fails — exactly as it should
try:
    assert_aligned(reaches.to_crs(4326), subbasins, hs_path)
except ValueError as e:
    print("✗ caught:", e)

In [ ]:
# [R] Layering terra rasters under sf vectors with tmap

library(terra)
library(sf)
library(tmap)

# Hillshade from the DEM — terra has this built in
slp <- terrain(r, "slope",  unit = "radians")
asp <- terrain(r, "aspect", unit = "radians")
hs  <- shade(slp, asp, angle = 45, direction = 315)
names(hs) <- "hillshade"

# ── The defensive habit, as a function — steal this ─────────────────────────
assert_aligned <- function(...) {
  layers <- list(...)
  codes  <- vapply(layers, function(l) {
    e <- if (inherits(l, "SpatRaster")) terra::crs(l, describe = TRUE)$code else st_crs(l)$epsg
    as.character(e)
  }, character(1))
  if (length(unique(codes)) > 1)
    stop("CRS mismatch: ", paste(codes, collapse = " / "))
  message("OK - all ", length(layers), " layers aligned on EPSG:", codes[1])
  invisible(codes[1])
}

# ✓ RIGHT — check alignment FIRST, then draw raster, then vectors
assert_aligned(reaches, subbasins, hs)

tmap_mode("plot")
tm_shape(hs) +
  tm_raster(palette = grey(0:100 / 100), legend.show = FALSE, alpha = 0.55) +
tm_shape(subbasins) +
  tm_borders(col = "grey45", lty = "dashed") +
tm_shape(reaches) +
  tm_lines(col = "priority", palette = "viridis", style = "fixed",
           breaks = c(0, 20, 40, 60, 80, 100), lwd = 2,
           colorNA = "#C9524A", title.col = "Habitat priority") +
tm_layout(title = "Hillshade, then vectors on top", frame = FALSE,
          legend.position = c("right", "bottom"))

# NOTE: terra/tmap will often reproject on the fly, which is friendlier than
# matplotlib — but do not rely on it. Reproject explicitly when it matters:
# hs_ll <- project(hs, "EPSG:4326")

### 🔧 Try it yourself — layering

1. Swap the draw order: plot `reaches` before the hillshade. Where did they go? Now you know what a z-order
   bug looks like, so you won't spend an hour blaming the data.
2. Change hillshade `alpha` from 0.55 to 1.0. At what alpha does the terrain start competing with your
   reaches instead of supporting them?
3. Change the hillshade azimuth from 315° to 135° (light from the southeast). Ridges will read as valleys —
   your brain assumes light from the upper-left. That's a *perceptual* bug that no test will catch.
4. Wire `assert_aligned()` into the top of the Section 5 map. Make it a reflex.

---
## Section 7 — Interactive Web Maps

Same layer, same breaks, now in a browser. One HTML file you can email to anyone — no licence, no install,
no ArcGIS Online credit.

### The four libraries, and the honest way to think about them

Each language gives you the same pair: one library that hands you a map in a single line, and one that
gives you control over every element. **They are not competitors — they're different moments in your day.**

| | Fast look (one line) | Full control (production) |
|---|---|---|
| **Python** | `leafmap` | `folium` |
| **R** | `mapview` | `leaflet` |

- **`leafmap` / `mapview`** — for *you*. Checking whether the join worked. Confirming the reaches land in
  the right basin. Batteries included: basemaps, popups, layer control, raster support, all by default.
- **`folium` / `leaflet`** — for *everyone else*. Explicit tooltips, controlled palettes, deliberate zoom
  bounds, a legend you wrote.

> ⚠ **The mistake to avoid:** `mapview` and `leafmap` are so convenient that people ship them. Interactive
> defaults are for exploration. A public deliverable needs decisions, not defaults — which means dropping
> to `folium` or `leaflet` once you know what you're showing.

### The rule that breaks everything

**Reproject to EPSG:4326 first.** Web maps assume lon/lat. `folium` and `leaflet` will not do it for you,
and they **fail silently** — the layer just doesn't appear, or lands off the coast of Africa at (0, 0).
This is the single most common web-mapping support question, in both languages.

### And the honest trade-off

Interactive is not strictly better than static. It can't be printed, it's harder to archive, and it invites
zoom levels your data doesn't support. **A 1:24,000 layer zoomed to 1:1,000 is a lie you handed the
public.** Set `min_zoom` / `max_zoom` on purpose.

### 7a — `folium`: explicit control (Python)

`folium` wraps Leaflet.js. Nothing is automatic, which is exactly why it's the right tool for a deliverable.

Two things that catch everyone:
- `folium.Choropleth` needs `key_on` to match a property in the GeoJSON — the string is
  `"feature.properties.<column>"` and getting it wrong produces a silently unstyled map.
- Tooltips are what make a map *stand alone*. Without one, a reader sees a colour and can't read a value.

In [ ]:
# [Python] folium — the deliverable version, with every decision made on purpose

import folium

# ── RULE 1: reproject to 4326 BEFORE anything else ──────────────────────────
reaches_ll   = reaches.to_crs(4326)
subbasins_ll = subbasins.to_crs(4326)
print(f"reaches CRS: {reaches.crs.to_epsg()}  ->  web CRS: {reaches_ll.crs.to_epsg()}")

centre = [reaches_ll.geometry.centroid.y.mean(), reaches_ll.geometry.centroid.x.mean()]

m = folium.Map(location=centre, zoom_start=10, tiles="CartoDB positron",
               min_zoom=8, max_zoom=13)   # deliberate: don't let them zoom past the data's honesty

# ── Colour by the SAME fixed breaks as the static map ───────────────────────
import branca.colormap as cm
BINS = [0, 20, 40, 60, 80, 100]
palette = cm.StepColormap(["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"],
                          vmin=0, vmax=100, index=BINS,
                          caption="Habitat priority (0–100, WDFW 2024 method)")

def style_fn(feat):
    v = feat["properties"]["priority"]
    return {"color": "#B0B0B0" if v is None else palette(v),
            "weight": 4, "opacity": 0.9}

folium.GeoJson(
    reaches_ll,
    name="Reach priority",
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["reach_id", "subbasin", "priority", "spawners", "barriers"],
        aliases=["Reach:", "Subbasin:", "Priority (0–100):", "Spawners:", "Barriers:"],
        localize=True, sticky=False),
    popup=folium.GeoJsonPopup(fields=["reach_id", "subbasin", "priority"],
                              aliases=["Reach", "Subbasin", "Priority"]),
).add_to(m)

folium.GeoJson(subbasins_ll, name="Subbasins",
               style_function=lambda f: {"color": "#4B2E83", "weight": 1.5,
                                         "fillOpacity": 0.04, "dashArray": "5,5"},
               tooltip=folium.GeoJsonTooltip(fields=["subbasin", "reach_count"],
                                             aliases=["Subbasin:", "Reaches:"])).add_to(m)

palette.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

folium_path = os.path.join(OUTDIR, "reach_priority_folium.html")
m.save(folium_path)
print(f"saved: {folium_path}  ({os.path.getsize(folium_path):,} bytes)")
print("One self-contained HTML file. Email it, or drop it on any web server.")
m   # renders inline in Jupyter

In [ ]:
# [Python] folium.Choropleth — the key_on trap, shown working

sb_ll = subbasins.to_crs(4326).copy()
sb_ll["spawners_per_km2"] = (sb_ll["spawners"] / subbasins["area_km2"]).round(1)

m2 = folium.Map(location=centre, zoom_start=9, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=sb_ll.to_json(),
    data=sb_ll,
    columns=["subbasin", "spawners_per_km2"],
    key_on="feature.properties.subbasin",   # <- the string that trips everyone
    fill_color="YlGnBu", fill_opacity=0.75, line_opacity=0.4,
    nan_fill_color="lightgrey",
    legend_name="Spawners per km² (normalized — see Section 3)",
).add_to(m2)

# Choropleth alone has no tooltips — add a transparent layer on top for hover
folium.GeoJson(sb_ll, style_function=lambda f: {"fillOpacity": 0, "weight": 0},
               tooltip=folium.GeoJsonTooltip(
                   fields=["subbasin", "spawners", "spawners_per_km2"],
                   aliases=["Subbasin:", "Spawners (count):", "Per km²:"])).add_to(m2)

m2.save(os.path.join(OUTDIR, "subbasin_density_folium.html"))
print("key_on must be 'feature.properties.<column>' — a typo here silently unstyles the map.")
print("Note we mapped the NORMALIZED value, not the raw count. Section 3 applies on the web too.")
m2

### 7b — `leafmap`: batteries included (Python)

`leafmap` (Qiusheng Wu, 2021) is the Python analogue of R's `mapview`: a map in one line, with basemaps,
layer control, and popups already wired up. It's the fastest way to *look* at spatial data in a notebook.

**Two backends — and the choice matters:**

| Import | Backend | Use when |
|---|---|---|
| `import leafmap` | `ipyleaflet` | Interactive work *in* the notebook. Needs Jupyter widgets. Richer drawing tools. |
| `import leafmap.foliumap as leafmap` | `folium` | You need a **standalone HTML export**, or widgets aren't available (Colab, some Hubs, nbconvert). |

> **On JupyterHub, prefer `leafmap.foliumap`.** The ipyleaflet backend needs the widget extension enabled;
> the folium backend just works and exports clean HTML. Same API either way, which is the nice part.

What `leafmap` gives you that raw `folium` doesn't:
- `add_gdf()` — a GeoDataFrame on the map with popups, one line
- `add_raster()` — a **local GeoTIFF** on a web map, correctly warped (this is the headline feature)
- `split_map()` — a before/after slider, ~3 lines
- `add_basemap("...")` — dozens of named basemaps without hunting for tile URLs

In [ ]:
# [Python] leafmap — the one-liner that replaces twenty lines of folium

# On JupyterHub use the folium backend: exports clean HTML, no widget extension needed.
import leafmap.foliumap as leafmap

m3 = leafmap.Map(center=centre, zoom=10)
m3.add_basemap("CartoDB.Positron")

# One line. Popups, layer control, zoom-to-layer — all wired up for you.
m3.add_gdf(reaches_ll, layer_name="Reach priority", info_mode="on_hover")
m3.add_gdf(subbasins_ll, layer_name="Subbasins",
           style={"color": "#4B2E83", "fillOpacity": 0.04, "weight": 1.5})

leafmap_path = os.path.join(OUTDIR, "reach_priority_leafmap.html")
m3.to_html(leafmap_path)
print(f"saved: {leafmap_path}  ({os.path.getsize(leafmap_path):,} bytes)")
print("\nCompare this cell to the folium cell above. Same map, a fraction of the code.")
print("That's the trade: leafmap decided the details for you. Fine for YOU. Think before you ship it.")
m3

In [ ]:
# [Python] leafmap's headline feature — a local GeoTIFF on a web map

# add_raster() reprojects your UTM GeoTIFF to web mercator and serves it as tiles.
# Requires: localtileserver + rioxarray. On JupyterHub it also needs jupyter-server-proxy.
# It is genuinely the easiest way to get a raster onto a slippy map in Python.

m4 = leafmap.Map(center=centre, zoom=10)
m4.add_basemap("CartoDB.DarkMatter")

try:
    m4.add_raster(dem_path, colormap="terrain", layer_name="Elevation (m)", opacity=0.75)
    m4.add_gdf(reaches_ll, layer_name="Reaches", info_mode="on_hover")
    m4.to_html(os.path.join(OUTDIR, "dem_leafmap.html"))
    print("✓ add_raster worked — the DEM is on a slippy map, reprojected for you.")
except Exception as e:
    print(f"✗ add_raster unavailable here ({type(e).__name__}: {str(e)[:70]})")
    print("  This is the trade-off: leafmap's raster support depends on localtileserver,")
    print("  which needs a tile server the browser can reach. Behind a strict proxy it fails.")
    print("  Fallback below uses folium.ImageOverlay — fewer dependencies, less capable.")

m4

In [ ]:
# [Python] Dependency-free raster fallback — ImageOverlay from a PNG
# Works anywhere folium works. Use when add_raster can't reach a tile server.

import matplotlib
from rasterio.warp import calculate_default_transform, reproject, Resampling

with rasterio.open(hs_path) as src:
    dst_crs = "EPSG:4326"
    transform, w, h = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
    arr = np.empty((h, w), dtype="uint8")
    reproject(source=rasterio.band(src, 1), destination=arr,
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear)
    west, south, east, north = rasterio.transform.array_bounds(h, w, transform)

png_path = os.path.join(OUTDIR, "hillshade_4326.png")
matplotlib.image.imsave(png_path, arr, cmap="Greys_r")

m5 = folium.Map(location=centre, zoom_start=10, tiles="CartoDB positron")
folium.raster_layers.ImageOverlay(
    image=png_path, bounds=[[south, west], [north, east]],
    opacity=0.55, name="Hillshade").add_to(m5)
folium.GeoJson(reaches_ll, style_function=style_fn,
               tooltip=folium.GeoJsonTooltip(fields=["reach_id", "priority"])).add_to(m5)
folium.LayerControl().add_to(m5)
m5.save(os.path.join(OUTDIR, "hillshade_overlay_folium.html"))

print(f"reprojected {arr.shape} hillshade to 4326 and overlaid as a PNG")
print("Note we had to reproject the raster BY HAND. That's what add_raster was doing for you.")
m5

In [ ]:
# [Python] leafmap split_map — the before/after slider, in three lines
# Genuinely hard to build by hand; nearly free here. Great for a public comment site.

m6 = leafmap.Map(center=centre, zoom=10)
m6.split_map(left_layer="OpenTopoMap", right_layer="CartoDB.Positron")
m6.to_html(os.path.join(OUTDIR, "split_leafmap.html"))
print("Drag the slider. For 'before restoration / after restoration' this is the whole feature.")
m6

### 7c — `leaflet`: explicit control (R)

R's `leaflet` is the direct analogue of `folium` — same JavaScript library underneath, same philosophy:
you make every decision. The pipe (`|>`) makes the layer stack read top to bottom.

Same rule as Python: **`st_transform(4326)` first.**

In [ ]:
# [R] leaflet — the deliverable version, every decision made on purpose

library(leaflet)
library(sf)

# ── RULE 1: reproject to 4326 BEFORE anything else ─────────────────────────
reaches_ll   <- st_transform(reaches, 4326)
subbasins_ll <- st_transform(subbasins, 4326)
cat("reaches CRS:", st_crs(reaches)$epsg, " ->  web CRS:", st_crs(reaches_ll)$epsg, "\n")

# Same fixed breaks as the static map — the legend must mean the same thing
BINS <- c(0, 20, 40, 60, 80, 100)
pal  <- colorBin("viridis", domain = reaches_ll$priority, bins = BINS,
                 na.color = "#B0B0B0")

leaflet(reaches_ll,
        options = leafletOptions(minZoom = 8, maxZoom = 13)) |>   # deliberate zoom bounds
  addProviderTiles(providers$CartoDB.Positron) |>
  addPolygons(data = subbasins_ll, color = "#4B2E83", weight = 1.5,
              fillOpacity = 0.04, dashArray = "5,5",
              label = ~subbasin, group = "Subbasins") |>
  addPolylines(color = ~pal(priority), weight = 4, opacity = 0.9,
               label = ~paste0("Reach ", reach_id, " (", subbasin, "): ",
                               ifelse(is.na(priority), "not scored", priority)),
               popup = ~paste0("<b>Reach ", reach_id, "</b><br>",
                               "Subbasin: ", subbasin, "<br>",
                               "Priority: ", priority, "<br>",
                               "Spawners: ", spawners, "<br>",
                               "Barriers: ", barriers),
               group = "Reach priority") |>
  addLegend(pal = pal, values = ~priority, opacity = 0.9,
            title = "Habitat priority<br>(0-100, WDFW 2024)",
            position = "bottomright") |>
  addLayersControl(overlayGroups = c("Reach priority", "Subbasins"),
                   options = layersControlOptions(collapsed = FALSE))

# Save a standalone HTML — the R equivalent of folium's m.save()
# library(htmlwidgets)
# saveWidget(m, file.path(tempdir(), "reach_priority_leaflet.html"), selfcontained = TRUE)

### 7d — `mapview`: batteries included (R)

`mapview` is the R analogue of `leafmap` — and it is the single best ergonomic in either language for
*looking at* spatial data. `mapview(reaches)` and you have an interactive map with a basemap, a popup
containing every attribute, and zoom-to-layer. No arguments required.

**Where it earns its place:**
- **Instant QA.** Did the join work? Are the reaches in the right basin? One line, one look.
- **The `+` operator.** `mapview(a) + mapview(b)` stacks layers with a layer control for free.
- **It reads rasters too.** `mapview(dem_raster)` just works.
- **`mapview()` on an `sf` object shows the attribute table in the popup** — which is often exactly what
  you need mid-debug.

> ⚠ **Do not ship it.** `mapview` picks the palette, the breaks, the basemap, and the popup contents *for*
> you. Those are the four decisions this whole module has been about. It's a debugging tool that happens to
> be pretty — which is precisely the trap. When the map is for someone else, move to `leaflet`.

**`tmap` has a third way:** `tmap_mode("view")` turns the *static map you already wrote* interactive with
one line. Same object, same breaks, now zoomable. If you built your map in `tmap`, this is the shortest
honest path to the browser.

In [ ]:
# [R] mapview — instant QA, one line

library(mapview)
library(sf)
library(terra)

# ── The one-liner. This is the whole point. ────────────────────────────────
mapview(reaches)

# Colour by an attribute — still one line
mapview(reaches, zcol = "priority")

# Control what you get, when you need to — same fixed breaks as the static map
mapview(reaches,
        zcol        = "priority",
        at          = c(0, 20, 40, 60, 80, 100),
        col.regions = viridisLite::viridis,
        lwd         = 4,
        layer.name  = "Habitat priority",
        na.color    = "#B0B0B0")

# ── The + operator: stack layers, get a layer control free ─────────────────
mapview(subbasins, zcol = "subbasin", alpha.regions = 0.15, legend = FALSE) +
  mapview(reaches, zcol = "priority", at = c(0, 20, 40, 60, 80, 100), lwd = 4)

# ── It reads rasters too ───────────────────────────────────────────────────
mapview(r, layer.name = "Elevation (m)") +
  mapview(reaches, zcol = "priority", lwd = 3)

# ── QA in practice — this is what mapview is actually for ──────────────────
mapview(reaches, zcol = "subbasin")                        # did the join work?
mapview(reaches[is.na(reaches$priority), ], color = "red", lwd = 5)   # where are the holes?

# ── Global options — set once per session ─────────────────────────────────
# mapviewOptions(basemaps = c("CartoDB.Positron", "Esri.WorldImagery"),
#                legend.pos = "bottomright")

# ── Export. mapshot() writes standalone HTML or a PNG snapshot ────────────
# m <- mapview(reaches, zcol = "priority")
# mapshot(m, url  = file.path(tempdir(), "reach_priority_mapview.html"))  # interactive
# mapshot(m, file = file.path(tempdir(), "reach_priority_mapview.png"))   # needs webshot2

# ── tmap's third way: the static map you already wrote, now interactive ───
# library(tmap)
# tmap_mode("view")
# tm_shape(reaches) +
#   tm_lines(col = "priority", palette = "viridis",
#            style = "fixed", breaks = c(0, 20, 40, 60, 80, 100), lwd = 3)
# tmap_mode("plot")      # switch back to static

### 🔧 Try it yourself — interactive maps

1. **Break it on purpose.** Delete `.to_crs(4326)` from the folium cell and re-run. The map renders, the
   tiles load, and your data is nowhere. No error. Now you'll recognise this in five seconds instead of
   an hour.
2. Set `max_zoom=18` on the folium map and zoom all the way in. Your reach geometry was digitized at
   roughly 1:24,000. At z18 you're implying sub-metre precision you do not have. Is that map honest?
3. Compare `leafmap`'s `add_gdf()` output to the `folium` cell. List three decisions leafmap made for you.
   For each, decide whether you'd accept it in a public deliverable.
4. In R: run `mapview(reaches)` then the full `leaflet` pipeline. Time yourself. Then ask which one you'd
   put on the agency's public comment site, and why the answer isn't the fast one.
5. `split_map()` with `OpenTopoMap` vs `Esri.WorldImagery`. For a restoration before/after, what would
   your two layers be?

---
## Section 8 — ArcPy Layouts and the R-ArcGIS Bridge

You already make every decision in this module — you make them in the **Symbology pane**. `arcpy.mp` is
that pane serialized into lines you can diff, loop, and put under version control.

Look at what these two lines are:

```python
sym.renderer.classificationMethod = "NaturalBreaks"
sym.renderer.breakCount = 5
```

That is exactly `style = "jenks", n = 5` in tmap. Same sentence, different accent.

**So why do it in `arcpy.mp` rather than clicking?** One word: **batch**. Twelve subbasins, twelve identical
layouts, one loop, at 300 dpi, re-runnable when the data changes. That's not a cartography trick — it's the
entire argument of this course, and Module 7 is where it starts to pay.

| Task | ArcGIS Pro (`arcpy.mp`) | Open source |
|---|---|---|
| Apply graduated colours | `sym.renderer.classificationMethod` | `scheme=` / `style=` |
| Export a layout at 300 dpi | `lyt.exportToPNG(path, resolution=300)` | `fig.savefig(dpi=300)` / `tmap_save()` |
| Publish a web map | `arcpy.SharePackage` → AGOL (licence) | `m.save()` → any web server (free) |
| Batch 12 basins | `for` loop over `lyt.exportToPNG` | `for` loop over `savefig` |

> **Environment note:** ArcPy only runs in the ArcGIS Pro Python environment (or a clone). These cells will
> not run on JupyterHub — they're here for reference and for lab use inside ArcGIS Pro's notebook.

In [ ]:
# [Python / ArcPy] Drive symbology and layout export from code
# Run inside ArcGIS Pro's Notebook or a cloned arcgispro-py3 environment.

try:
    import arcpy

    # ── Open a project and grab the map ─────────────────────────────────────
    # aprx = arcpy.mp.ArcGISProject(r'C:\Projects\Nooksack\Habitat.aprx')
    # m    = aprx.listMaps('Priority Map')[0]
    # lyr  = m.listLayers('reach_priority')[0]

    # ── Graduated colours — this IS the Symbology pane, in text ─────────────
    # sym = lyr.symbology
    # sym.updateRenderer('GraduatedColorsRenderer')
    # sym.renderer.classificationField  = 'priority'
    # sym.renderer.classificationMethod = 'NaturalBreaks'   # or 'EqualInterval', 'Quantile', 'DefinedInterval'
    # sym.renderer.breakCount = 5
    # ramp = aprx.listColorRamps('Viridis')[0]
    # sym.renderer.colorRamp = ramp
    # lyr.symbology = sym

    # ── Manual breaks — the defensible choice for a comparison map ──────────
    # sym.renderer.classificationMethod = 'ManualInterval'
    # breaks = [20, 40, 60, 80, 100]
    # for i, brk in enumerate(sym.renderer.classBreaks):
    #     brk.upperBound = breaks[i]
    #     brk.label = f'{breaks[i-1] if i else 0} to {breaks[i]}'
    # lyr.symbology = sym

    # ── Export the layout at print resolution ───────────────────────────────
    # lyt = aprx.listLayouts('Report Layout')[0]
    # lyt.exportToPNG(r'C:\Projects\Nooksack\out\priority_map.png', resolution=300)
    # lyt.exportToPDF(r'C:\Projects\Nooksack\out\priority_map.pdf')

    # ── THE PAYOFF: twelve basins, one loop, identical cartography ──────────
    # for basin in ['North Fork', 'Middle Fork', 'South Fork', 'Mainstem']:
    #     lyr.definitionQuery = f"subbasin = '{basin}'"
    #     m.defaultCamera.setExtent(arcpy.Describe(lyr).extent)
    #     for elm in lyt.listElements('TEXT_ELEMENT', 'Title'):
    #         elm.text = f'Habitat Priority — {basin}'
    #     lyt.exportToPNG(rf'C:\Projects\Nooksack\out\{basin.replace(" ", "_")}.png',
    #                     resolution=300)
    # print('Four layouts exported — identical symbology, identical breaks, comparable.')

    print("ArcPy is available — uncomment the lines above and point them at a real .aprx.")

except ModuleNotFoundError:
    print("ArcPy not available in this environment — expected on JupyterHub.")
    print("These cells are reference material; run them inside ArcGIS Pro's Notebook.")
    print()
    print("The line worth remembering:")
    print("    sym.renderer.classificationMethod = 'NaturalBreaks'")
    print("    sym.renderer.breakCount = 5")
    print("...is the same decision as tmap's  style = 'jenks', n = 5")

In [ ]:
# [R] R-ArcGIS Bridge — LOCAL ONLY (needs ArcGIS Pro on the same machine)
# Will NOT run on JupyterHub. Use RStudio or Positron on a machine with Pro installed.

# library(arcgisbinding)
# arc.check_product()          # confirms the licence and Pro install
#
# # ── Read a feature class straight out of a File GDB into sf ───────────────
# # fc      <- arc.open("C:/Projects/Nooksack/Habitat.gdb/reach_priority")
# # arc_df  <- arc.select(fc, fields = c("reach_id", "subbasin", "priority"))
# # reaches <- arc.data2sf(arc_df)
#
# # ── Do the cartography in tmap, where you have real control ───────────────
# # library(tmap)
# # m <- tm_shape(reaches) +
# #        tm_lines(col = "priority", palette = "viridis",
# #                 style = "fixed", breaks = c(0, 20, 40, 60, 80, 100), lwd = 2)
# # tmap_save(m, "priority_map.png", dpi = 300)
#
# # ── Write results back to the GDB for the Pro users on your team ──────────
# # arc.write("C:/Projects/Nooksack/Habitat.gdb/reach_priority_v2", reaches,
# #           overwrite = TRUE)
#
# # ── arcgislayers: the modern alternative for ArcGIS Online / Portal ───────
# # library(arcgislayers)
# # flayer  <- arc_open("https://services.arcgis.com/.../FeatureServer/0")
# # reaches <- arc_select(flayer, fields = c("reach_id", "priority"))
#
# # The pattern: bridge the DATA out, do the CARTOGRAPHY in tmap/ggplot2,
# # bridge the RESULT back. Don't try to drive Pro's layout engine from R.

cat("R cell — LOCAL ONLY. Needs ArcGIS Pro + arcgisbinding on the same machine.\n")
cat("Will not run on JupyterHub. Run in RStudio or Positron locally.\n")

---
## Section 9 — Lab: Build the Report Map

**Scenario:** It's Tuesday. The salmon recovery board meets at 1 pm. They need one map in the printed report
and one map on the public comment site. Your Module 6 priority scores are done and defensible.

Work in groups of 3–4. Steps 1–4 are the core; step 5 is for groups running ahead.

| Step | Task | The point |
|---|---|---|
| **1** | Load `reaches` + `subbasins` + hillshade. Run `assert_aligned()` before you plot anything. | Most stalls happen here, not later. |
| **2** | Pick your symbology. Reaches by `priority`; subbasins by *something normalized*, not a raw count. | Section 3. |
| **3** | Try three classification methods. Pick one. **Write the sentence** that defends it. | Section 4. This is the actual deliverable. |
| **4** | Add the furniture: legend with units, NoData class, title, scale bar, credits with CRS. Export at 300 dpi. | Section 5. |
| **5** | Publish an interactive version — `folium`/`leafmap` or `leaflet`/`mapview`. Set `max_zoom` on purpose. | Section 7. |

> **The standard is not "does it run."** The standard is: **would you put your name on this?**
> Export at 300 dpi and *actually open the PNG*. Screen-legible is not print-legible — 8 pt legend text
> disappears on paper.

**Known failure points** (so you don't lose 20 minutes):
- `scheme=` needs `mapclassify` installed — the `ImportError` is unhelpful.
- `contextily` basemaps are a network call. Behind an agency firewall they fail. Skip the basemap; it isn't the point.
- `tm_scale_bar()` (tmap v3) was renamed `tm_scalebar()` (v4). Check `packageVersion("tmap")`.
- Forgetting `.to_crs(4326)` before folium/leaflet. Silent. Always.

In [ ]:
# [Python] Lab — your starting scaffold. Fill in the TODOs.

# ── STEP 1: load + verify ───────────────────────────────────────────────────
assert_aligned(reaches, subbasins, hs_path)
print(f"{len(reaches)} reaches | {int(reaches['priority'].isna().sum())} unscored | CRS {reaches.crs.to_epsg()}")

# ── STEP 2: choose symbology ────────────────────────────────────────────────
sb_lab = subbasins.copy()
sb_lab["spawners_per_km2"] = sb_lab["spawners"] / sb_lab["area_km2"]
# TODO: decide what the subbasin fill should encode. Not a raw count.

# ── STEP 3: choose classification — and defend it ───────────────────────────
CANDIDATES = {
    "natural_breaks": {"scheme": "natural_breaks", "k": 5},
    "quantile":       {"scheme": "quantiles", "k": 5},
    "policy":         {"scheme": "user_defined",
                       "classification_kwds": {"bins": [20, 40, 60, 80, 100]}},
}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, kw) in zip(axes, CANDIDATES.items()):
    reaches.plot(ax=ax, column="priority", cmap="viridis", linewidth=1.8,
                 missing_kwds={"color": "0.8"}, **kw)
    ax.set_title(name, loc="left"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

CHOICE = "policy"      # TODO: change this to your pick
DEFENCE = (
    "I used manual breaks at 20/40/60/80 because the board is comparing these reaches "
    "against the 2020 baseline, and recomputed breaks would make the two maps "
    "incomparable even though both would look correct."
)
# TODO: rewrite DEFENCE in your own words. If it says "because it looked best," try again.
print(f"\nCHOICE: {CHOICE}\nDEFENCE: {DEFENCE}")

In [ ]:
# [Python] Lab — STEP 4: the report map, with all the furniture

fig, ax = plt.subplots(figsize=(11, 11))

with rasterio.open(hs_path) as src:
    rshow(src, ax=ax, cmap="Greys_r", alpha=0.5)
sb_lab.plot(ax=ax, facecolor="none", edgecolor="0.5", linewidth=0.8, linestyle="--")
reaches.plot(ax=ax, column="priority", cmap="viridis", **CANDIDATES[CHOICE],
             linewidth=2.4, legend=True,
             legend_kwds={"title": "Habitat priority\n(0–100, WDFW 2024 method)",
                          "loc": "lower right", "fontsize": 8.5, "title_fontsize": 9.5,
                          "frameon": True, "framealpha": 0.92},
             missing_kwds={"color": "#C9524A", "label": "Not scored (4 reaches)"})

ax.set_title("Salmon Habitat Priority — Nooksack Basin", fontsize=16, fontweight="bold",
             loc="left", pad=12)
ax.text(0, 1.005, f"Restoration prioritization · {CHOICE} classification · 2024 assessment",
        transform=ax.transAxes, fontsize=10, color="0.4")
add_scale_bar(ax, 10000)
add_north_arrow(ax)
add_credits(ax, "Data: WDFW 2024 (synthetic teaching data) · CRS: EPSG:32610 (UTM 10N) · "
                "Analysis: GISPR Module 6 · Cartography: GISPR Module 7")
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)

lab_png = os.path.join(OUTDIR, "LAB_report_map_300dpi.png")
plt.savefig(lab_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"exported: {lab_png} ({os.path.getsize(lab_png):,} bytes)")
print("\nNow OPEN it at 100%. Can you read the legend? The scale bar label? The credits?")
print("If not, the map is not done — regardless of whether the code ran.")

In [ ]:
# [R] Lab — the same five steps in R

library(sf)
library(terra)
library(tmap)

# ── STEP 1: load + verify ──────────────────────────────────────────────────
assert_aligned(reaches, subbasins, hs)
cat(nrow(reaches), "reaches |", sum(is.na(reaches$priority)), "unscored | CRS:",
    st_crs(reaches)$epsg, "\n")

# ── STEP 2: symbology — normalize before you map ───────────────────────────
subbasins$spawners_per_km2 <- subbasins$spawners / subbasins$area_km2
# TODO: decide what the subbasin fill should encode. Not a raw count.

# ── STEP 3: three candidates, pick one, defend it ──────────────────────────
tmap_mode("plot")
tmap_arrange(
  tm_shape(reaches) + tm_lines(col = "priority", style = "jenks", n = 5,
                               palette = "viridis", lwd = 2) +
    tm_layout(title = "jenks", frame = FALSE, legend.show = FALSE),
  tm_shape(reaches) + tm_lines(col = "priority", style = "quantile", n = 5,
                               palette = "viridis", lwd = 2) +
    tm_layout(title = "quantile", frame = FALSE, legend.show = FALSE),
  tm_shape(reaches) + tm_lines(col = "priority", style = "fixed",
                               breaks = c(0, 20, 40, 60, 80, 100),
                               palette = "viridis", lwd = 2) +
    tm_layout(title = "policy", frame = FALSE, legend.show = FALSE),
  nrow = 1)

CHOICE  <- "policy"    # TODO: change this to your pick
DEFENCE <- paste("I used manual breaks at 20/40/60/80 because the board is comparing",
                 "these reaches against the 2020 baseline, and recomputed breaks would",
                 "make the two maps incomparable even though both would look correct.")
# TODO: rewrite DEFENCE in your own words. "Because it looked best" is not a defence.
cat("\nCHOICE:", CHOICE, "\nDEFENCE:", DEFENCE, "\n")

# ── STEP 4: the report map ─────────────────────────────────────────────────
tm_scalebar_c <- if (utils::packageVersion("tmap") >= "4.0.0") tmap::tm_scalebar else tmap::tm_scale_bar

m <- tm_shape(hs) +
       tm_raster(palette = grey(0:100 / 100), legend.show = FALSE, alpha = 0.5) +
     tm_shape(subbasins) +
       tm_borders(col = "grey50", lty = "dashed") +
     tm_shape(reaches) +
       tm_lines(col = "priority", palette = "viridis", style = "fixed",
                breaks = c(0, 20, 40, 60, 80, 100), lwd = 2.4,
                title.col = "Habitat priority\n(0-100, WDFW 2024 method)",
                colorNA = "#C9524A", textNA = "Not scored (4 reaches)") +
     tm_scalebar_c(breaks = c(0, 5, 10), position = c("left", "bottom")) +
     tm_compass(type = "arrow", position = c("right", "top")) +
     tm_credits("Data: WDFW 2024 (synthetic) | EPSG:32610 | GISPR Modules 6-7",
                size = 0.5, position = c("left", "bottom")) +
     tm_layout(title = "Salmon Habitat Priority - Nooksack Basin",
               frame = FALSE, legend.position = c("right", "bottom"))
m

lab_png <- file.path(tempdir(), "LAB_report_map_300dpi.png")
tmap_save(m, lab_png, dpi = 300, width = 8, height = 8)
cat("\nexported:", lab_png, "\n")
cat("Now OPEN it at 100%. Can you read the legend? The scale bar label? The credits?\n")
cat("If not, the map is not done - regardless of whether the code ran.\n")

# ── STEP 5: the interactive version — the SAME object, one line ────────────
# tmap_mode("view")
# m
# tmap_mode("plot")
#
# # or with mapview for a fast look:
# # mapview(reaches, zcol = "priority", at = c(0, 20, 40, 60, 80, 100), lwd = 4)

---
## Section 10 — Extended Application: Wrap It in a Function

**Scenario extension:** the board now wants the same map for each of the four subbasins, plus a basin-wide
overview. Five maps, identical cartography, comparable legends.

This is the payoff. Not "matplotlib can draw a line" — but *the Tuesday request takes 30 seconds instead of
3 hours, and every map is identical because the same code drew it.*

`map_reaches()` below is a **carry-forward function**: it joins `inspect_raster()` and `raster_pipeline()`
from Modules 5–6 in your personal toolkit. Module 8 turns it into the last cell of an end-to-end pipeline.

In [ ]:
# [Python] A reusable, parameterized map function — carry this into Module 8

def map_reaches(reaches, subbasins=None, hillshade_path=None, *,
                column="priority", bins=(20, 40, 60, 80, 100), cmap="viridis",
                title="Habitat Priority", subtitle=None, extent=None,
                credits="Data: WDFW 2024 (synthetic) · CRS: EPSG:32610 · GISPR Module 7",
                out_path=None, dpi=300, figsize=(11, 11)):
    """Render a report-ready reach priority map.

    Fixed `bins` by default — so every map this function draws is comparable to
    every other one. That is the entire reason it takes `bins` and not `scheme`.

    Returns the matplotlib Figure.
    """
    if subbasins is not None:
        assert_aligned(reaches, subbasins)

    fig, ax = plt.subplots(figsize=figsize)

    if hillshade_path:
        with rasterio.open(hillshade_path) as src:
            rshow(src, ax=ax, cmap="Greys_r", alpha=0.5)
    if subbasins is not None:
        subbasins.plot(ax=ax, facecolor="none", edgecolor="0.5",
                       linewidth=0.8, linestyle="--")

    reaches.plot(ax=ax, column=column, cmap=cmap, scheme="user_defined",
                 classification_kwds={"bins": list(bins)}, linewidth=2.4, legend=True,
                 legend_kwds={"title": f"{column.replace('_',' ').title()}\n(0–100, WDFW 2024 method)",
                              "loc": "lower right", "fontsize": 8.5, "title_fontsize": 9.5,
                              "frameon": True, "framealpha": 0.92},
                 missing_kwds={"color": "#C9524A",
                               "label": f"Not scored ({int(reaches[column].isna().sum())})"})

    if extent is not None:
        ax.set_xlim(extent[0], extent[2]); ax.set_ylim(extent[1], extent[3])

    ax.set_title(title, fontsize=16, fontweight="bold", loc="left", pad=12)
    if subtitle:
        ax.text(0, 1.005, subtitle, transform=ax.transAxes, fontsize=10, color="0.4")
    add_scale_bar(ax, 10000); add_north_arrow(ax); add_credits(ax, credits)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

    if out_path:
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    return fig


# ── THE PAYOFF: five maps, identical cartography, one loop ──────────────────
targets = [("Basin overview", None)] + [(sb, sb) for sb in sorted(reaches["subbasin"].unique())]

for label, sb in targets:
    sel = reaches if sb is None else reaches[reaches["subbasin"] == sb]
    ext = None if sb is None else sel.buffer(2500).total_bounds
    p = os.path.join(OUTDIR, f"map_{label.replace(' ', '_').lower()}.png")
    fig = map_reaches(sel, subbasins=subbasins, hillshade_path=hs_path,
                      title=f"Salmon Habitat Priority — {label}",
                      subtitle=f"{len(sel)} reaches · fixed breaks 20/40/60/80 · 2024 assessment",
                      extent=ext, out_path=p, dpi=150)
    plt.close(fig)
    print(f"  ✓ {label:16} -> {os.path.basename(p):38} {os.path.getsize(p):>8,} bytes")

print("\nFive maps. Identical breaks, identical legend, identical furniture.")
print("Re-run it next year when the data changes. That is the whole argument of this course.")

# Show one of them
fig = map_reaches(reaches[reaches["subbasin"] == "South Fork"], subbasins=subbasins,
                  hillshade_path=hs_path, title="Salmon Habitat Priority — South Fork",
                  subtitle="8 reaches · fixed breaks 20/40/60/80 · 2024 assessment",
                  extent=reaches[reaches["subbasin"] == "South Fork"].buffer(2500).total_bounds,
                  figsize=(9, 9))
plt.show()

In [ ]:
# [R] The same reusable function in R — carry this into Module 8

library(tmap)
library(sf)

map_reaches <- function(reaches, subbasins = NULL, hs = NULL,
                        column   = "priority",
                        breaks   = c(0, 20, 40, 60, 80, 100),
                        palette  = "viridis",
                        title    = "Habitat Priority",
                        credits  = "Data: WDFW 2024 (synthetic) | EPSG:32610 | GISPR Module 7",
                        out_path = NULL, dpi = 300) {

  # Fixed `breaks` by default — so every map this function draws is comparable
  # to every other one. That is the entire reason it takes breaks, not style.
  if (!is.null(subbasins)) stopifnot(st_crs(reaches) == st_crs(subbasins))

  tm_scalebar_c <- if (utils::packageVersion("tmap") >= "4.0.0") {
    tmap::tm_scalebar
  } else {
    tmap::tm_scale_bar
  }

  m <- NULL
  if (!is.null(hs)) {
    m <- tm_shape(hs) +
         tm_raster(palette = grey(0:100 / 100), legend.show = FALSE, alpha = 0.5)
  }
  if (!is.null(subbasins)) {
    sb_layer <- tm_shape(subbasins) + tm_borders(col = "grey50", lty = "dashed")
    m <- if (is.null(m)) sb_layer else m + sb_layer
  }

  n_na <- sum(is.na(reaches[[column]]))
  reach_layer <- tm_shape(reaches) +
    tm_lines(col = column, palette = palette, style = "fixed", breaks = breaks,
             lwd = 2.4, colorNA = "#C9524A",
             textNA = paste0("Not scored (", n_na, ")"),
             title.col = "Habitat priority\n(0-100, WDFW 2024 method)")
  m <- if (is.null(m)) reach_layer else m + reach_layer

  m <- m +
    tm_scalebar_c(breaks = c(0, 5, 10), position = c("left", "bottom")) +
    tm_compass(type = "arrow", position = c("right", "top")) +
    tm_credits(credits, size = 0.5, position = c("left", "bottom")) +
    tm_layout(title = title, frame = FALSE, legend.position = c("right", "bottom"))

  if (!is.null(out_path)) tmap_save(m, out_path, dpi = dpi, width = 8, height = 8)
  m
}

# ── THE PAYOFF: five maps, identical cartography, one loop ─────────────────
outdir <- tempdir()
for (sb in c("North Fork", "Middle Fork", "South Fork", "Mainstem")) {
  sel <- reaches[reaches$subbasin == sb, ]
  p   <- file.path(outdir, paste0("map_", gsub(" ", "_", tolower(sb)), ".png"))
  map_reaches(sel, subbasins, hs,
              title    = paste("Salmon Habitat Priority -", sb),
              out_path = p, dpi = 150)
  cat(sprintf("  OK %-14s -> %s\n", sb, basename(p)))
}
cat("\nFour maps. Identical breaks, identical legend, identical furniture.\n")
cat("Re-run it next year when the data changes. That is the whole argument.\n")

# Show one
map_reaches(reaches[reaches$subbasin == "South Fork", ], subbasins, hs,
            title = "Salmon Habitat Priority - South Fork")

### 🔧 Try it yourself — the reusable function

1. Add a `basemap=True` argument that calls `contextily.add_basemap()` — and handle the network failure
   gracefully, because an agency firewall will block it.
2. Add a `comparison_safe=True` guard that **refuses** to run if someone passes `scheme=` instead of
   `bins=`. Make the function enforce the lesson.
3. Add an `interactive=True` branch that returns a `folium` map instead of a Figure — same breaks, same
   colours, same legend text. One function, both media.
4. Point it at one of your own agency's layers. What breaks? That's your Module 8 backlog.

---

## 🔍 Module 7 — Self-Check Questions

Work through these without scrolling up. Then verify by running code.

**Symbology**
1. Why is a choropleth of a raw count almost always wrong? What are the *two* correct alternatives?
2. You have a land-cover layer with 8 classes. Why is `cmap="viridis"` the wrong choice?
3. Roughly 1 in 12 men has some form of colour vision deficiency. What does that mean for a red-green
   map in a public meeting of 60 people — and what's your obligation as a public agency?

**Classification**
4. Explain in one sentence why Jenks is wrong for a two-map comparison, even though it's "optimal."
5. What does `mapclassify.Quantiles(vals, k=5).counts` return, and why does that explain why quantile maps
   always look good?
6. Write the GeoPandas argument that pins breaks at 20/40/60/80. Now the tmap equivalent.

**Cartographic elements**
7. Your priority field has 4 NaN reaches out of 44. What happens to them if you omit `missing_kwds`, and
   what does a reader conclude?
8. Which two elements make a map *verifiable* rather than merely attractive? Defend your pick.
9. When is a north arrow redundant? When is a scale bar redundant?

**Layering and CRS**
10. Two layers in different CRS, both plotted to the same matplotlib Axes. What warning do you get? What
    does the figure look like?
11. Write the assertion you'd put at the top of any multi-layer plotting script.

**Interactive**
12. What single line must come before `folium.GeoJson(gdf)`, and what happens if you forget it?
13. `key_on="feature.properties.subbasin"` — what breaks if you typo it, and how would you notice?
14. When would you reach for `leafmap`/`mapview` over `folium`/`leaflet`? Give the one-sentence rule.
15. Your reaches were digitized at 1:24,000. Why is `max_zoom=18` an honesty problem, not a technical one?
16. Which `leafmap` backend exports standalone HTML without Jupyter widgets, and why does that matter on
    JupyterHub?

---
## ✅ Module 7 Homework — Deliverables

Submit the following in Canvas before the next session.

### Deliverable 1 — The Report Map (25 pts)
Produce **one publication-quality static map** of a result layer — the Nooksack reaches, or your own
agency data:
- Symbology matched to the measurement level (normalize anything that's a count)
- A classification you chose deliberately, with the NoData class visible
- Full furniture: legend with units, title, scale bar, credits naming source, vintage, CRS, and author
- Exported at **300 dpi**

Submit: the PNG, **and one sentence** completing *"I used ___ breaks because ___."*
If that sentence says "because it looked best," you haven't finished the deliverable.

### Deliverable 2 — The Interactive Map (25 pts)
Publish the **same layer, with the same breaks**, to the browser:
- Python: `folium` or `leafmap` · R: `leaflet` or `mapview`
- Reprojected to EPSG:4326
- Tooltips or popups that let the map stand alone without you next to it
- `min_zoom` / `max_zoom` set deliberately for your data's actual precision

Submit: the standalone HTML file, plus **two sentences** on what you gained and what you gave up versus the
static version.

### Deliverable 3 — The Comparison, Automated (50 pts)
Adapt `map_reaches()` (or write your own) to produce **at least four comparable maps** in a loop:
- Fixed, identical breaks across every panel — and a comment explaining why they must be
- At least one raster layer beneath the vectors, CRS-checked with an assertion before the first plot call
- Parameterized: the function takes the subset and the output path as arguments
- Both a static export and one interactive version

Then write a short markdown cell: **which two maps would you take to the board, and what do you say when
someone asks why the other version looks different?** That paragraph is worth more than the code.

Commit the notebook and outputs to your GitHub repo (same repo as Modules 2–6). Tag the commit
`module7-deliverable`.

---

**Stuck?** Post in Ed Discussion — tag `#module7`. Include your error message, the cell that failed, and
your OS/environment. If you solved something tricky, share how — the whole cohort benefits.

---

### Coming next — Module 8: End-to-End Workflows

Tonight the map was the destination. In Module 8 it becomes **the last cell of a pipeline** — raw data to
finished PNG in one script, re-runnable when the data changes. You'll bring `map_reaches()` with you.